# intradaySignal research driver

薄 driver：每個訊號一個設定檔 `signals/<name>.py`（四段：envelope+condition / point_filter / day_selector / model），
註冊在 `config.py`；四層各一個模組在 `layers/`。這裡只讀各層產物、訓練、選單、進 BackTest。

前置（每日增量，或研究時手動）：
```bash
uv run python src/research/intradaySignal/pipeline/run.py all --only negfill
# 或分層
uv run python src/research/intradaySignal/pipeline/run.py envelope       # 1 envelope 掃描 → store → 滾動 condition 參數 → index
uv run python src/research/intradaySignal/pipeline/run.py points         # 2 固定 tick 過濾 → 候選點
uv run python src/research/intradaySignal/pipeline/run.py day_selector   # 3 學習型 stock-day gate → day scores
uv run python src/research/intradaySignal/pipeline/run.py model          # 4 Ridge A/M + 16-cell 門檻 → model/<st>_selected
```

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', '..', '..', '..')))

import pandas as pd
import polars as pl
pd.set_option('display.max_columns', None)

from src.research.intradaySignal.config import REGISTRY, resolve_dirs
from src.research.intradaySignal.framework.store import store_dates
from src.research.intradaySignal.layers.envelope import read_condition_params, read_signal_index, winner_history
from src.research.intradaySignal.layers.point_filter import read_points
from src.research.intradaySignal.layers.day_selector import read_day_scores
from src.research.intradaySignal.layers.model import (
    SIGNAL_COL, build_signal_frame, train_by_signal, threshold_by_signal, select_by_signal,
)
from src.research.qcut_analyzer import QcutAnalyzer
from src.research.backTest import BackTest

DIRS = resolve_dirs(None)
print(REGISTRY.names, '| store days:', len(store_dates(DIRS['store'])))

## 1. 各層產物：index（L1）→ points（L2）→ day scores（L3）

In [ ]:
for name in REGISTRY.names:
    sig = REGISTRY.get(name)
    index = read_signal_index(DIRS['index'], sig); points = read_points(DIRS['points'], sig); scores = read_day_scores(DIRS['params'], sig)
    scored = scores.filter(pl.col('reason') == 'scored') if scores.height else scores
    print(f"{name:8s} index {index.height:>10,} rows / {index['Date'].n_unique():>4} days | points {points.height:>10,} rows"
          f" | day scores {scores.height:>7,} stock-days, kept share {scored['kept'].mean() if scored.height else float('nan'):.3f}")
    display(winner_history(read_condition_params(DIRS['params'], sig)).tail(3))

## 2. signal_df：long format，`signal_id` 區分訊號

進模型的列 = store ∩ index ∩ points ∩ day selector 保留的 stock-day；`use_day_selector=False` 可看沒過 L3 的版本。

In [ ]:
N_DAYS = 230
signal_df = build_signal_frame(REGISTRY, dirs=DIRS, last_n_days=N_DAYS)
signal_df.groupby(SIGNAL_COL).agg(rows=('Date', 'size'), days=('Date', 'nunique'), stocks=('QuoteCode', 'nunique'))

## 3. 分訊號訓練（A: timing 60d / M: selection 20d）與 16-cell 門檻

In [ ]:
results = train_by_signal(signal_df, REGISTRY)
thresholded = threshold_by_signal(results, REGISTRY)
for name, res in thresholded.items():
    print(name, 'rows', len(res.df), 'pass_threshold', int(res.df['pass_threshold'].sum()), '| use_model_gate =', REGISTRY.get(name).model.use_model_gate)

In [ ]:
spec = REGISTRY.get('negfill').model; df = thresholded['negfill'].df
QcutAnalyzer.plot_qcut_analysis(df, spec.timing_pred_col, spec.timing_target, date_col='Date', split_date='2026-01-01', q=8)

## 3b. 訊號相似度（Phase 2 四把尺）

exact tick／stock-day／±N 秒鄰近／日 label 相關，以及 A 的 disjoint 子集 label。可比任一層：points、過門檻後、最終選單。

In [ ]:
from src.research.intradaySignal.framework.overlap import overlap_report

def _pl(df, label):
    return pl.from_pandas(df[['Date', 'QuoteCode', 'ChannelSeq', 'TransTime', label]].reset_index(drop=True)).with_columns(pl.col('ChannelSeq').cast(pl.UInt64), pl.col('Date').cast(pl.String))

if {'fh_neg', 'negfill'} <= set(thresholded):
    lab = REGISTRY.get('negfill').model.raw_label_col
    a = _pl(thresholded['fh_neg'].df, lab)                                                # fh: static book = every row
    b = _pl(thresholded['negfill'].df[thresholded['negfill'].df.pass_threshold], lab)
    display(overlap_report(a, b, a_name='fh_neg', b_name='negfill', label_col=lab).as_table().to_pandas())

## 4. 最終選單（過門檻；靜態 book 全留）→ BackTest 候選

In [ ]:
selected = select_by_signal(thresholded, REGISTRY)
for name, df in selected.items():
    sd = df.groupby(['Date', 'QuoteCode'])[REGISTRY.get(name).model.raw_label_col].mean()
    print(f"{name:8s} selected rows={len(df):>8,}  stock-days={len(sd):>7,}  sd-mean BP={sd.mean():7.1f}")
signal_df_model = selected['negfill'].copy()

In [ ]:
# 部位簿記（execution 層，與訊號無關）
signal_df_model['PosLots'] = signal_df_model.groupby(['Date', 'QuoteCode']).BidPrice1.transform('cumcount') + 1
signal_df_model['Position'] = signal_df_model.groupby(['Date', 'QuoteCode']).BidPrice1.transform('cumsum') / 10
signal_df_model['accLots'] = signal_df_model.groupby(['Date', 'QuoteCode']).BidPrice1.transform('cumcount') + 1
posLimit = 500
signalwithRisk_df = signal_df_model[
    (signal_df_model.accLots < signal_df_model.avg_askLots1 + signal_df_model.avg_bidLots1)
    & (signal_df_model.Position < posLimit)
    & (signal_df_model.RemainSeconds >= 3600 + 60 * 25)
].reset_index(drop=True)
signalwithRisk_df['Date'] = pd.to_datetime(signalwithRisk_df.Date).dt.strftime('%Y%m%d')
len(signalwithRisk_df)

In [ ]:
bt = BackTest(signalwithRisk_df[(signalwithRisk_df.Date.astype(int) > 20260101)], hedge=True)
results_bt = bt.run_analysis()
daily_ticker_df = bt.get_daily_ticker_result()